## Read-in raw files and down-sample

In [ ]:
import os
import glob
import pandas as pd
import re

### List of files to process

In [104]:
path = "/Users/andy/_data/Google Local" # Read-from and write-to this directory

file_list = glob.glob(os.path.join(path, '*.json.gz'))
file_list

['/Users/andy/_data/Google Local/review-Florida_10.json.gz',
 '/Users/andy/_data/Google Local/review-Georgia_10.json.gz',
 '/Users/andy/_data/Google Local/review-Indiana_10.json.gz',
 '/Users/andy/_data/Google Local/review-Delaware_10.json.gz',
 '/Users/andy/_data/Google Local/review-California_10.json.gz',
 '/Users/andy/_data/Google Local/review-Arkansas_10.json.gz',
 '/Users/andy/_data/Google Local/review-Alaska_10.json.gz',
 '/Users/andy/_data/Google Local/review-Connecticut_10.json.gz',
 '/Users/andy/_data/Google Local/review-Arizona_10.json.gz',
 '/Users/andy/_data/Google Local/review-Illinois_10.json.gz',
 '/Users/andy/_data/Google Local/review-District_of_Columbia_10.json.gz',
 '/Users/andy/_data/Google Local/review-Idaho_10.json.gz',
 '/Users/andy/_data/Google Local/review-Kansas_10.json.gz',
 '/Users/andy/_data/Google Local/review-Alabama_10.json.gz',
 '/Users/andy/_data/Google Local/review-Iowa_10.json.gz',
 '/Users/andy/_data/Google Local/review-Colorado_10.json.gz',
 '/User

### Temporarily pick 2

In [105]:
file_list = file_list[6], file_list[10] # Test with a small file, Alaska
file_list

('/Users/andy/_data/Google Local/review-Alaska_10.json.gz',
 '/Users/andy/_data/Google Local/review-District_of_Columbia_10.json.gz')

### Stratify down-sample (currently at 1%)

And save to same folder as .csv

In [ ]:
for file in file_list:
    print(f' "{file}" ')
    df = pd.read_json(file, compression='gzip', lines=True, dtype={'text': 'str'})

    # Clean up
    df = df[['rating','text']] # Keep relevant columns
    df = df[df['text'].apply(lambda x: isinstance(x, str))] # Drop non-string instances
    df = df[ df['text'] != 'None' ] # Drop blank entries

    # Calculate stratified sample sizes by rating class
    sample_size = int(round(df.shape[0] * 0.01)) # 1% sample size
    group_counts = df['rating'].value_counts(normalize=True)
    group_samples = (group_counts * sample_size).astype(int)
    group_samples # Number of samaples to draw for each rating class

    # Stratified sample by rating class
    df_list = []
    for strata in range(1,df['rating'].max()+1):
        df_strata = df[ df['rating'] == strata ]
        df_strata = df_strata.sample(group_samples[strata], random_state=100)
        df_list.append(df_strata)

    df_sampled = pd.concat(df_list, ignore_index=True)

    print(f'Original: {df_sampled.shape[0]} \n Sampled: {df.shape[0]}') # Close

    state_name = re.search(r'review-(.+?)_10', file).group(1)
    df_sampled.to_csv(f'{path}/{state_name}.csv.gz', index=False, compression='gzip')


 "/Users/andy/_data/Google Local/review-Alaska_10.json.gz" 
Original: 2980 
 Sampled: 298257
 "/Users/andy/_data/Google Local/review-District_of_Columbia_10.json.gz" 
Original: 2942 
 Sampled: 294459


### Read-in saved .csv files

In [114]:
file_list_csv = glob.glob(os.path.join(path, '*.csv.gz'))
file_list_csv

['/Users/andy/_data/Google Local/District_of_Columbia.csv.gz',
 '/Users/andy/_data/Google Local/Alaska.csv.gz']

In [115]:
df_csv_list = []
for file in file_list_csv:
    file = file_list_csv[1]
    df_csv = pd.read_csv(file)
    df_csv_list.append(df_csv)

df_all_states = pd.concat(df_csv_list, ignore_index=True)


In [116]:
df_all_states.shape

(5960, 2)